In [ ]:
# Import required packages.
import pandas as pd 
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import matplotlib.image as mpimg
import geopandas as gpd 
from sklearn.preprocessing import StandardScaler
import argparse
from hmmlearn.hmm import GaussianHMM
import folium
from folium.plugins import HeatMap

cows_f = pd.read_csv("Cows_Feb24_to_May25.csv") # # Read in the raw data.

np.random.seed(0) # Set seed for reproducibility.

In [ ]:
# Average number of raw gps recordings per animal over the time period.
print(cows_f.groupby("Serial").size().mean())

In [ ]:
# Assign variables to an object.
continuous1 = ["HDOP", "Horizontal Accuracy", "#Satellites"]

# Calculate descriptive statistics for numeric variables.
numeric1 = cows_f[continuous1].describe()
for column in continuous1:
    numeric1.loc['median',column] = cows_f[column].median()
print(numeric1)

In [ ]:
# Filter dataframe by individual animal.
c400 = cows_f.loc[cows_f['Serial'] == 295400]
c400

In [ ]:
# Rename columns to remove spaces and hashtags.
# Create a new copy of the specified columns with a different name.
cows_f["Pasture_ID"] = cows_f["Pasture ID"]
cows_f["Pasture_Version"] = cows_f["Pasture Version"]
cows_f["Message_Type"] = cows_f["Message Type"]
cows_f["Battery_Voltage"] = cows_f["Battery Voltage"]
cows_f["GSM_RAT"] = cows_f["GSM RAT"]
cows_f["GSM_Operator"] = cows_f["GSM Operator"]
cows_f["Position_Time"] = cows_f["Position Time"]
cows_f["Horizontal_Accuracy"] = cows_f["Horizontal Accuracy"]
cows_f["n_Satellites"] = cows_f["#Satellites"]
cows_f["Fence_Status"] = cows_f["Fence Status"]
cows_f["Collar_Status"] = cows_f["Collar Status"]

# Remove the originally named column so only the renamed column remains.
cows_f = cows_f.drop('Pasture ID', axis=1)
cows_f = cows_f.drop('Pasture Version', axis=1)
cows_f = cows_f.drop('Message Type', axis=1)
cows_f = cows_f.drop('Battery Voltage', axis=1)
cows_f = cows_f.drop('GSM RAT', axis=1)
cows_f = cows_f.drop('GSM Operator', axis=1)
cows_f = cows_f.drop('Position Time', axis=1)
cows_f = cows_f.drop('Horizontal Accuracy', axis=1)
cows_f = cows_f.drop('#Satellites', axis=1)
cows_f = cows_f.drop('Fence Status', axis=1)
cows_f = cows_f.drop('Collar Status', axis=1)

# Convert column to datetime.
cows_f["Position_Time"] = pd.to_datetime(cows_f["Position_Time"]) 

# Filters applied to account for off-site activity, location inaccuracies caused by poor satellite geometry and hardware limitations.
cows_f.loc[cows_f['Latitude'] > 52.645, 'Latitude'] = np.nan
cows_f.loc[cows_f['Latitude'] < 52.620, 'Latitude'] = np.nan
cows_f.loc[cows_f['Longitude'] > 1.28, 'Longitude'] = np.nan
cows_f.loc[cows_f['Longitude'] < 1.24, 'Longitude'] = np.nan
cows_f.loc[cows_f['Horizontal_Accuracy'] > 150, 'Horizontal_Accuracy'] = np.nan
cows_f.loc[cows_f['Collar_Status'] == "OffAnimal", 'Collar_Status'] = np.nan
cows_f.loc[cows_f['HDOP'] > 200, 'HDOP'] = np.nan
cows_f.loc[cows_f['n_Satellites'] < 5, 'n_Satellites'] = np.nan

# Remove columns.
cows_f = cows_f.drop('Activity', axis=1)
cows_f = cows_f.drop('Mode', axis=1)
cows_f = cows_f.drop('Battery_Voltage', axis=1)
cows_f = cows_f.drop('GSM_RAT', axis=1)
cows_f = cows_f.drop('GSM_Operator', axis=1)

# Remove the remaining rows with missing values for the respective fields.
cows_f.dropna(inplace=True)

# Convert data type from float to integer (discrete).
cows_f["Pasture_ID"]=cows_f["Pasture_ID"].astype(int)
cows_f["Pasture_Version"]=cows_f["Pasture_Version"].astype(int)
cows_f["n_Satellites"]=cows_f["n_Satellites"].astype(int)

In [ ]:
# Assign variables to an object.
continuous1 = ["HDOP", "Horizontal_Accuracy", "n_Satellites"]

# Calculate descriptive statistics for numeric variables.
numeric1 = cows_f[continuous1].describe()
for column in continuous1:
    numeric1.loc['median',column] = cows_f[column].median()
print(numeric1)

In [ ]:
# Convert latitude/longitude to projected coordinates.
gdf = gpd.GeoDataFrame( 
    cows_f, 
    geometry=gpd.points_from_xy(cows_f["Longitude"], cows_f["Latitude"]), 
    crs="EPSG:4326"
) 
 
# Convert to British National Grid CRS. 
gdf = gdf.to_crs("EPSG:27700") 
 
gdf["x"] = gdf.geometry.x
gdf["y"] = gdf.geometry.y

In [ ]:
# Calculate movement features.
def add_movement_features(cows_f): 
    cows_f = cows_f.sort_values(["Serial", "Position_Time"]).copy() 
 
    cows_f["x_prev"] = cows_f.groupby("Serial")["x"].shift(1) 
    cows_f["y_prev"] = cows_f.groupby("Serial")["y"].shift(1) 
    cows_f["t_prev"] = cows_f.groupby("Serial")["Position_Time"].shift(1) 
 
    dx = cows_f["x"] - cows_f["x_prev"] 
    dy = cows_f["y"] - cows_f["y_prev"] 
 
    cows_f["step_length_m"] = np.sqrt(dx**2 + dy**2)
    cows_f["time_delta_min"] = (cows_f["Position_Time"] - cows_f["t_prev"]).dt.total_seconds() / 60 
    cows_f["speed_m_per_min"] = cows_f["step_length_m"] / cows_f["time_delta_min"] 
 
    cows_f["heading"] = np.arctan2(dy, dx) 
    cows_f["heading_prev"] = cows_f.groupby("Serial")["heading"].shift(1) 
    turn = cows_f["heading"] - cows_f["heading_prev"] 
    cows_f["turning_angle"] = np.arctan2(np.sin(turn), np.cos(turn))
    cows_f["abs_turning_angle"] = np.abs(cows_f["turning_angle"]) 
 
    cows_f["hour"] = cows_f["Position_Time"].dt.hour 
    cows_f["date"] = cows_f["Position_Time"].dt.date 
    cows_f["month"] = cows_f["Position_Time"].dt.month 
    cows_f["day_of_year"] = cows_f["Position_Time"].dt.dayofyear 
 
    return cows_f
 
gdf = add_movement_features(gdf) 

gdf.to_csv("cow_movement_features.csv", index=False) 

In [ ]:
# Ensure data is datetime format and sorted by cow and time.
df = gdf.sort_values(by=["Serial", "Position_Time"]).reset_index(drop=True)

# Calculate the time gap between the previous record for the same cow.
df["gap_seconds"] = df.groupby("Serial")["Position_Time"].diff().dt.total_seconds()

# Define the 15-minute target window (+/- 30 seconds = 870 to 930 seconds).
df["is_15min_target"] = df["gap_seconds"].between(870, 930)

# Define the wider 10-20 min window (600 to 1200 seconds).
df["is_10_20min_gap"] = df["gap_seconds"].between(600, 1200)

# Calculate consecutive run IDs for the 10-20 min window per cow.
df["run_id"] = (df["is_10_20min_gap"] != df.groupby("Serial")["is_10_20min_gap"].shift()).cumsum()


# Categorise records into parts of the day.
def get_day_part(hour):
    if 5 <= hour < 12: return "Morning"
    elif 12 <= hour < 17: return "Afternoon"
    elif 17 <= hour < 21: return "Evening"
    else: return "Night"
df["day_part"] = df["Position_Time"].dt.hour.apply(get_day_part)


# Aggregate by cow and day.
def analyze_day(group):
    # Counts and times.
    num_records = len(group)
    first_time = group["Position_Time"].min()
    last_time = group["Position_Time"].max()
    
    # Spans and Gaps.
    day_span_hours = (last_time - first_time).total_seconds() / 3600.0
    largest_gap_min = group["gap_seconds"].max() / 60.0 if num_records > 1 else 0.0
    
    # Rates.
    records_per_hour = num_records / day_span_hours if day_span_hours > 0 else num_records
    
    # Target 15-min intervals (+/- 30s).
    num_15min_records = group["is_15min_target"].sum()
    
    # Expected records.
    expected_records = (day_span_hours * 4) + 1
    pct_expected_captured = (num_15min_records / expected_records) * 100 if expected_records > 1 else 0.0
    
    # Records by day part.
    day_part_counts = group["day_part"].value_counts().to_dict()
    
    # Calculate maximum consecutive points 10-20 mins apart.
    valid_runs = group[group["is_10_20min_gap"]]
    if not valid_runs.empty:
        # Group by run_id & count intervals to find the maximum consecutive point streak.
        # Adding 1 converts the number of gap into the number of GPS points.
        max_consec_points = int(valid_runs.groupby("run_id").size().max() + 1)
    else:
        max_consec_points = 0

    return pd.Series({
        "n_records": num_records,
        "number_15_minute_(+/-_30_seconds)": int(num_15min_records),
        "max_consecutive_10-20_mins_ap": max_consec_points, # <-- NEW VARIABLE
        "first_record": first_time.strftime("%H:%M:%S"),
        "last_record": last_time.strftime("%H:%M:%S"),
        "day_span_(hours)": round(day_span_hours, 2),
        "largest_gap_minutes": round(largest_gap_min, 2) if not pd.isna(largest_gap_min) else 0.0,
        "records_per_hour": round(records_per_hour, 2),
        "records_by_day_part": str(day_part_counts),
        "%_expected_15_minute_(+/-_30_seconds)_records_captured": round(pct_expected_captured, 1) 
    })

# Run the aggregation.
summary_table1 = df.groupby(["Serial", "date"], as_index=False).apply(analyze_day)

# Save dataframe.
summary_table1.to_csv("all_data_summary_table.csv", index=False)

# Display dataframe.
summary_table1

In [ ]:
# Filter summary table dataframe.
valid_groups = summary_table1[summary_table1["max_consecutive_10-20_mins_ap"] >= 0]

# Extract the Serial and date columns that match the filter condition.
valid_keys = valid_groups[["Serial", "date"]]

# Filter dataframe by matching serial and date with the valid keys.
df["date"] = df["Position_Time"].dt.date # Ensure dataframe has the date column extracted from Position_Time.
filtered_df = df.merge(valid_keys, on=["Serial", "date"], how="inner")

filtered_df # Display filtered dataframe

In [ ]:
# Remove rows where the movement calculation is not meaningful, such as the first row for each animal or very large gaps between fixes. 
clean = filtered_df.dropna( 
    subset=["step_length_m", "time_delta_min", "speed_m_per_min", "turning_angle"] 
).copy() 

# Keep roughly 15-minute fixes, allowing 30 seconds tolerance. 
clean = clean[(clean["time_delta_min"] >= 14.5) & (clean["time_delta_min"] <= 15.5)] 

# Remove implausibly high speeds after inspecting the distribution. 
clean = clean[clean["speed_m_per_min"] < 650] 

In [ ]:
# Descriptive statistics for the processed dataframe.
# Assign variables to an object.
continuous1 = ["speed_m_per_min", "abs_turning_angle", "step_length_m", "time_delta_min", "HDOP", "Horizontal_Accuracy", "n_Satellites"]

# Calculate descriptive statistics for numeric variables.
numeric1 = clean[continuous1].describe()
for column in continuous1:
    numeric1.loc['median',column] = clean[column].median()
print(numeric1)

In [ ]:
# Plot the distribution of movement features across time intervals (used to evaluate different time filters).
other_features = ["speed_m_per_min", "turning_angle"]

plt.figure(figsize=(12, 10))

for i, feature in enumerate(other_features):
    plt.subplot(2, 2, i + 1)
    plt.scatter("time_delta_min", feature, s=1, c="black", alpha=0.5, data=clean) # Simplified mpl.pyplot.scatter
    plt.ylabel(feature)
    plt.xlabel("time_delta_min")

plt.suptitle('Figure 5. Scatterplots of Movement Features', y=1.02)
plt.tight_layout()

plt.savefig("move_feat_by_time_delta(Position_Time_15_mins+-30).png", dpi=300, bbox_inches='tight') # Save figure.

plt.show() # Display figure.

In [ ]:
# Distribution of movement features.
other_features = ["speed_m_per_min", "turning_angle"]

plt.figure(figsize=(12, 10))

for i, col in enumerate(other_features):
    plt.subplot(2, 2, i + 1)
    sns.kdeplot(data=clean, x=col, fill=True, common_norm=False, alpha=.5, linewidth=0)
    plt.xlabel(col)
    plt.ylabel("frequency")
plt.suptitle('Figure 2. KDE plots of Movement Features', y=1.02)
plt.tight_layout()

plt.savefig("move_feat_KDE.png", dpi=300, bbox_inches='tight') # Save figure.

plt.show() # Display figure.

In [ ]:
# Average number of gps recordings per animal over the time period (processed dataframe).
print((clean).groupby("Serial").size().mean()) 

In [ ]:
# Number of gps recordings per animal over the time period (processed dataframe).
print((clean).groupby("Serial").size())

In [ ]:
# Calculate the number of records per day per individual.
daily1 = ( 
    clean.groupby(["Serial", "date"]) 
    .size()
    .reset_index(name="frequency") 
) 

daily1

In [ ]:
# Average number of records per day per individual.
print((daily1["frequency"]).mean()) 

In [ ]:
# Box plot distribution of number of gps recordings per day for each individual (Used to check for sampling differences between collars).
plt.figure(figsize=(7, 5))
sns.boxplot(x='Serial', y="frequency", data=daily1)
plt.title("Frequency of recordings per day")
plt.show()

In [ ]:
# Distribution of the number of GPs recordings by hour.
hourly1 = ( 
    clean.groupby(["Serial", "hour"]) 
    .size()
    .reset_index(name="frequency") 
) 

# Filters for individual animals.
d1 = hourly1.loc[hourly1['Serial'] == 294366]
d2 = hourly1.loc[hourly1['Serial'] == 295400]
d3 = hourly1.loc[hourly1['Serial'] == 295402]
d4 = hourly1.loc[hourly1['Serial'] == 295770]
d5 = hourly1.loc[hourly1['Serial'] == 295815]
d6 = hourly1.loc[hourly1['Serial'] == 295830]

# Barplot.
plt.figure(figsize=(5, 4))
sns.barplot(x='hour', y="frequency", data=hourly1)
plt.title("Frequency of recordings per hour")
plt.show()

In [ ]:
# Prepare features for Hidden Markov Model (HMM).
hmm_df = clean.copy() 
hmm_df["log_step"] = np.log1p(hmm_df["step_length_m"])
hmm_df["log_speed"] = np.log1p(hmm_df["speed_m_per_min"])
hmm_df["abs_turning_angle"] = np.abs(hmm_df["turning_angle"])

hmm_df = hmm_df.sort_values(["Serial", "Position_Time"]).copy()

hmm_df.to_csv("hmm_df.csv", index=False) # Save dataframe.

In [ ]:
# Total sum of time intervals per individual.
d1 = hmm_df.loc[hmm_df['Serial'] == 294366]
d2 = hmm_df.loc[hmm_df['Serial'] == 295400]
d3 = hmm_df.loc[hmm_df['Serial'] == 295402]
d4 = hmm_df.loc[hmm_df['Serial'] == 295770]
d5 = hmm_df.loc[hmm_df['Serial'] == 295815]
d6 = hmm_df.loc[hmm_df['Serial'] == 295830]

print((hmm_df["time_delta_min"]).sum()) 
print((d1["time_delta_min"]).sum()) 
print((d2["time_delta_min"]).sum()) 
print((d3["time_delta_min"]).sum()) 
print((d4["time_delta_min"]).sum()) 
print((d5["time_delta_min"]).sum()) 
print((d6["time_delta_min"]).sum()) 

In [ ]:
# Total distance travelled per individual (processed dataframe).
print((hmm_df["step_length_m"]).sum()) 
print((d1["step_length_m"]).sum()) 
print((d2["step_length_m"]).sum()) 
print((d3["step_length_m"]).sum()) 
print((d4["step_length_m"]).sum()) 
print((d5["step_length_m"]).sum()) 
print((d6["step_length_m"]).sum()) 

In [ ]:
# AIC / BIC / log likelihood testing for number of hidden cattle behaviour states across multiple seeds for HMMs.
# ***Preliminary model***
def fit_model(X, n_states, seed, n_iter):
    model = GaussianHMM(
        n_components=n_states,
        covariance_type="full",
        n_iter=n_iter,
        random_state=seed,
        tol=1e-3
    )

    model.fit(X)

    return {
        "n_states": n_states,
        "seed": seed,
        "log_likelihood": model.score(X),
        "AIC": model.aic(X),
        "BIC": model.bic(X),
        "converged": model.monitor_.converged,
        "model": model
    }


def plot_model_selection(best_by_states, output_prefix):
    import matplotlib.pyplot as plt
    import seaborn as sns

    sns.set_theme(style="ticks", context="talk")

    plot_df = best_by_states.melt(
        id_vars="n_states",
        value_vars=["AIC", "BIC", "log_likelihood"],
        var_name="metric",
        value_name="value"
    )

    fig, ax1 = plt.subplots(figsize=(10, 6))

    sns.lineplot(
        data=plot_df[plot_df["metric"].isin(["AIC", "BIC"])],
        x="n_states",
        y="value",
        hue="metric",
        marker="o",
        linewidth=2.5,
        ax=ax1
    )

    ax1.set_xlabel("Number of hidden states")
    ax1.set_ylabel("Information criterion\nlower is better")

    ax2 = ax1.twinx()

    ax1.grid(False)
    ax2.grid(False)

    sns.lineplot(
        data=plot_df[plot_df["metric"] == "log_likelihood"],
        x="n_states",
        y="value",
        marker="o",
        linewidth=2.5,
        color="black",
        label="log_likelihood",
        ax=ax2
    )

    ax2.set_ylabel("Log-likelihood\nhigher is better")

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()

    ax1.legend(
        lines1 + lines2,
        labels1 + labels2,
        frameon=False,
        loc="best"
    )

    ax2.get_legend().remove()

    sns.despine(right=False)
    fig.tight_layout()

    fig.savefig(f"{output_prefix}.model_selection.png", dpi=300)
    fig.savefig(f"{output_prefix}.model_selection.pdf")

    plt.close(fig)



def main():
    parser = argparse.ArgumentParser(
        description="Fit Gaussian HMMs across state numbers and random seeds."
    )

    parser.add_argument("--input", required=True, help="Input CSV file")
    parser.add_argument("--min_states", type=int, default=2)
    parser.add_argument("--max_states", type=int, default=6)
    parser.add_argument("--seeds", type=int, default=100)
    parser.add_argument("--n_iter", type=int, default=500)
    parser.add_argument(
        "--columns",
        nargs="+",
        default=["speed_m_min", "turn_angle_deg", "step_length_m"],
        help="Columns to use for HMM fitting"
    )
    # ***Select parameters.***
    args = parser.parse_args([
    "--input", "hmm_df.csv",
    "--min_states", "1",
    "--max_states", "6",
    "--seeds", "100", # Should be >= 100 (set to low value to run script faster).
    "--n_iter", "500",
    "--columns", "speed_m_per_min", "abs_turning_angle"
    ])

    df = pd.read_csv(args.input)

    X = df[args.columns].dropna()

    all_results = []
    all_models = []

    for n_states in range(args.min_states, args.max_states + 1):
        for seed in range(args.seeds):
            try:
                fit = fit_model(X, n_states, seed, args.n_iter)

                all_models.append(fit)

                all_results.append({
                    "n_states": fit["n_states"],
                    "seed": fit["seed"],
                    "log_likelihood": fit["log_likelihood"],
                    "AIC": fit["AIC"],
                    "BIC": fit["BIC"],
                    "converged": fit["converged"]
                })

            except Exception as e:
                all_results.append({
                    "n_states": n_states,
                    "seed": seed,
                    "log_likelihood": None,
                    "AIC": None,
                    "BIC": None,
                    "converged": False,
                    "error": str(e)
                })

    results = pd.DataFrame(all_results)
    results.to_csv("hmm_model_comparison_all_seeds.csv", index=False)

    best_by_states = (
        results.dropna(subset=["AIC"])
        .sort_values("AIC")
        .groupby("n_states")
        .first()
        .reset_index()
    )

    best_by_states.to_csv("hmm_model_comparison_best_by_state.csv", index=False)

    plot_model_selection(best_by_states, "hmm")

    best_fit = min(
        [m for m in all_models if m["converged"]],
        key=lambda x: x["AIC"]
    )

    best_model = best_fit["model"]
    states = best_model.predict(X)

    df.loc[X.index, "hmm_state"] = states
    df.to_csv("hmm_states_output.csv", index=False)

    print("\nBest models by AIC:")
    print(best_by_states)

if __name__ == "__main__":
    main()

In [ ]:
# ***Preliminary model***
# Fit the model (using each individual as a separate sequence) based on the AIC / BIC / log likelihood curve 'elbow'.
X_list = [] 
lengths = [] 
 
for Serial, sub in hmm_df.groupby("Serial"): 
    Xi = sub[["speed_m_per_min", "abs_turning_angle"]].values 
    X_list.append(Xi) 
    lengths.append(len(Xi)) 
 
X_all = np.vstack(X_list) 
scaler = StandardScaler() 
X_all = scaler.fit_transform(X_all) 
 
model = GaussianHMM( 
    n_components=4,
    covariance_type="full", 
    n_iter=500,
    random_state=43,
    tol=0.001,
) 

model.fit(X_all, lengths=lengths) 
hmm_df["hmm_state_raw"] = model.predict(X_all, lengths=lengths) 

In [ ]:
# Inspect the raw states before naming them.
state_summary = ( 
    hmm_df.groupby("hmm_state_raw") 
    .agg( 
        mean_step_m=("step_length_m", "mean"), 
        median_step_m=("step_length_m", "median"), 
        mean_speed=("speed_m_per_min", "mean"), 
        mean_abs_turn=("abs_turning_angle", "mean"), 
        n=("hmm_state_raw", "size") 
    )
    .reset_index() 
) 

print(state_summary) 

In [ ]:
# Label raw state numbers them (only after looking at the state summary). 
state_labels = { 
    0: "Intensive Foraging",
    1: "Resting",
    2: "Transit",
    3: "Extensive Foraging",
}

hmm_df["behaviour_state"] = hmm_df["hmm_state_raw"].map(state_labels) 
hmm_df.to_csv("hmm_df_b_R_g.csv", index=False) # Save the dataframe with the decoded behaviour states assigned.

In [ ]:
# Behaviour state speed distributions (additional practice only: this is based on the preliminary model, not the final model).
sns.kdeplot(data=hmm_df, x="speed_m_per_min", hue="behaviour_state", fill=True, common_norm=False, palette="Set1",
   alpha=.5, linewidth=0)

In [ ]:
# Behaviour state turning angle distributions (practice only).
sns.kdeplot(data=hmm_df, x="turning_angle", hue="behaviour_state", fill=True, common_norm=False, palette="Set1",
   alpha=.5, linewidth=0)

In [ ]:
# Behavioural profile: percentages of how an individual uses its time (practice only).
profile = ( 
    hmm_df.groupby(["Serial", "behaviour_state"]) #as_index=False
    .size() 
    .groupby(level=0) 
    .apply(lambda x: 100 * x / x.sum()) 
    #.reset_index(name="frequency")
) 

profile

In [ ]:
# NVC plant community map.
state_gdf = gpd.GeoDataFrame( 
    hmm_df, 
    geometry=gpd.points_from_xy(hmm_df["x"], hmm_df["y"]), 
    crs="EPSG:27700" 
)

habitat = gpd.read_file("NVC polygons.shp").to_crs(state_gdf.crs)

# Produces a basic plot to visualise the data.
habitat.plot(column="NVC", 
             legend=True,
             cmap="tab20b",
             figsize=(12, 10))

In [ ]:
# Spatial join NVC map and decoded behaviour states dataframe (practice only).
habitat_aligned = habitat[["NVC", "geometry"]].to_crs(state_gdf.crs) # Ensure habitat matches state_gdf CRS before joining.

# Run the spatial join using "intersects" (safest for points-in-polygons).
joined = gpd.sjoin(
    state_gdf,
    habitat_aligned,
    how="left",
    predicate="intersects"
)

joined.replace({"NVC": 'woodland'}, 'Woodland', inplace=True)

In [ ]:
# Mapping preliminary model behaviour states (practice only).
print(state_summary) # Displays behaviour state characteristics

base_whole = habitat.plot(color='white', edgecolor='black', figsize=(12, 12))

# Filters to apply.
#jf = joined.loc[joined['Serial'] == 294366]
#jf = joined.loc[joined['Collar_Status'] == "OffAnimal"]
#jf = joined.loc[joined['Fence_Status'] == "NotStarted"]
jf = joined.loc[joined['hmm_state_raw'] <= 3]
#jf = jf.loc[jf['Serial'] == 294366]
#jf = jf.loc[jf['pt_numeric'] > ]
#jf = jf.loc[jf['pt_numeric'] < 500]

# Plot onto habitat map.
joined.plot(column='behaviour_state',
                 legend=True, 
                 ax=base_whole, 
                 cmap='tab20b',
                 c="b",
                 alpha=0.6,
                 markersize=0.5)

plt.savefig("all_states.png") 

In [ ]:
# Folium interactive heatmap.
jf = joined.loc[joined['hmm_state_raw'] == 1]

geo_data = jf[['Latitude', 'Longitude']]

heat_data = geo_data[['Latitude', 'Longitude']].values.tolist()

map_center = [geo_data['Latitude'].mean(), geo_data['Longitude'].mean()]
m = folium.Map(location=map_center, zoom_start=16)

HeatMap(heat_data, radius=3, blur=1, max_zoom=12).add_to(m)

m